<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

</br>

In [21]:
# TODO 0: 실습을 위해 아래 패키지를 import하고, iris 데이터셋을 이진 분류용으로 준비해주세요.
import numpy as np
import seaborn as sns

# iris 데이터셋 로드
df = sns.load_dataset("iris")

# 수치형 피처 선택
feature_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
X = df[feature_cols].values
y = (df["species"] == "virginica").astype(int).values  # virginica → 1, 나머지 → 0

# 표준화
X_mean, X_std = X.mean(axis=0), X.std(axis=0)
X_std = np.where(X_std == 0, 1.0, X_std)
X = (X - X_mean) / X_std

# Train/Valid 분할 (80:20)
dataset_count = len(X)
shuffled_index = np.random.permutation(dataset_count)
cut_index = int(dataset_count * 0.8)
X_train, X_valid = X[shuffled_index[:cut_index]], X[shuffled_index[cut_index:]]
y_train, y_valid = y[shuffled_index[:cut_index]], y[shuffled_index[cut_index:]]

print(f"데이터: iris ({len(X)}개, 이진 분류: virginica vs 나머지)")
print(f"피처: {feature_cols}")
print(f"클래스 분포 - 0: {np.sum(y==0)}, 1: {np.sum(y==1)}")
print(f"Train: {len(X_train)}개, Valid: {len(X_valid)}개")

print(X_train)

데이터: iris (150개, 이진 분류: virginica vs 나머지)
피처: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
클래스 분포 - 0: 100, 1: 50
Train: 120개, Valid: 30개
[[ 6.74501145e-01  3.28414053e-01  8.76433123e-01  1.44883158e+00]
 [ 3.10997534e-01 -3.62176246e-01  5.35408562e-01  2.64141916e-01]
 [-5.37177559e-01  7.88807586e-01 -1.28338910e+00 -1.05217993e+00]
 [-1.50652052e+00  7.88807586e-01 -1.34022653e+00 -1.18381211e+00]
 [ 9.16836886e-01 -1.31979479e-01  3.64896281e-01  2.64141916e-01]
 [ 2.24968346e+00 -1.31979479e-01  1.33113254e+00  1.44883158e+00]
 [ 1.52267624e+00 -1.31979479e-01  1.21745768e+00  1.18556721e+00]
 [ 1.03800476e+00  9.82172869e-02  3.64896281e-01  2.64141916e-01]
 [-1.14301691e+00  9.82172869e-02 -1.28338910e+00 -1.31544430e+00]
 [-7.79513300e-01  7.88807586e-01 -1.34022653e+00 -1.31544430e+00]
 [-1.14301691e+00  1.24920112e+00 -1.34022653e+00 -1.44707648e+00]
 [ 1.15917263e+00 -1.31979479e-01  9.90107977e-01  1.18556721e+00]
 [ 4.32165405e-01  7.88807586e-01  9.332

</br>

# 학습 내용
>이번 장에서는 <strong>로지스틱 회귀(Logistic Regression)</strong>에 대해 학습합니다.</br></br>
>시그모이드 함수, Binary Cross-Entropy 손실, 그래디언트 계산을 학습해봅시다.

</br>

# 선형 회귀 vs 로지스틱 회귀
> 선형 회귀는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">연속값을 예측</mark>하고, 로지스틱 회귀는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">클래스 확률을 예측</mark>합니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">비교 항목</th>
      <th>선형 회귀</th>
      <th>로지스틱 회귀</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">문제 유형</td><td>회귀 (연속값 예측)</td><td>분류 (클래스 예측)</td></tr>
    <tr><td style="text-align:center">출력 범위</td><td>$(-\infty, +\infty)$</td><td>$[0, 1]$ (확률)</td></tr>
    <tr><td style="text-align:center">활성화 함수</td><td>없음 (항등 함수)</td><td>시그모이드</td></tr>
    <tr><td style="text-align:center">손실 함수</td><td>MSE</td><td>Binary Cross-Entropy</td></tr>
    <tr><td style="text-align:center">예시</td><td>집값 예측, 온도 예측</td><td>스팸 분류, 질병 진단</td></tr>
  </tbody>
</table>

</br>

# 시그모이드 함수 (Sigmoid Function)
> 선형 점수 $z$를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">$[0, 1]$ 범위의 확률</mark>로 변환하는 활성화 함수입니다.

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">$z$ 값</th>
      <th style="text-align:center">$\sigma(z)$</th>
      <th>해석</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">$-5$</td><td style="text-align:center">$\approx 0.007$</td><td>거의 확실히 클래스 0</td></tr>
    <tr><td style="text-align:center">$0$</td><td style="text-align:center">$0.5$</td><td>중립 (결정 경계)</td></tr>
    <tr><td style="text-align:center">$+5$</td><td style="text-align:center">$\approx 0.993$</td><td>거의 확실히 클래스 1</td></tr>
  </tbody>
</table>

In [ ]:
# TODO 1: sigmoid 함수를 정의해봅시다.

def sigmoid(z):

    # TODO 1-1: overflow 방지를 위해 z 값을 -500에서 500 사이의 값으로 제한해봅시다.
    z = np.clip(z, -500, 500) # -500~ 500 사이로 값을 제한하기
    
    # TODO 1-2: 계산 결과를 반환해봅시다.
    return 1.0/ (1.0+np.exp(-z))


0.9933071490757153


In [3]:
# TODO 2: z = [-5, -1, 0, 1, 5]에 대해 결과를 출력해봅시다.
z = np.array([-5,-1, 0, 1, 5])
print(sigmoid(z))

[0.00669285 0.26894142 0.5        0.73105858 0.99330715]


💡 Overflow 방지: `np.clip(z, -500, 500)`

> 시그모이드 함수 $\sigma(z) = \frac{1}{1 + e^{-z}}$에서 $e^{-z}$를 계산할 때,</br>
> $z$가 극단적인 값이면 **overflow**가 발생합니다.
>**$z = -1000$인 경우 (clip 없이):**
>$$e^{-z} = e^{-(-1000)} = e^{1000} = \infty \quad \text{(overflow!)}$$
>$$\sigma(-1000) = \frac{1}{1 + \infty} = \text{NaN (계산 불가)}$$
>**$z = -1000$인 경우 (clip 적용):**
>$$z = \text{np.clip}(-1000,\ -500,\ 500) = -500$$
>$$e^{-z} = e^{-(-500)} = e^{500} \approx 1.4 \times 10^{217} \quad \text{(매우 크지만 유한)}$$
>$$\sigma(-500) = \frac{1}{1 + e^{500}} \approx 0 \quad \text{(정상 계산)}$$
> `np.clip`은 값의 범위를 $[-500, 500]$으로 제한합니다.</br>
> $\sigma(-500) \approx 0$, $\sigma(500) \approx 1$이므로 결과 정확도에는 영향이 없습니다.

💡분류 판정
> $\sigma(z) \geq 0.5$이면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">클래스 1</mark>, $\sigma(z) < 0.5$이면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">클래스 0</mark>으로 분류합니다.</br>
> 이 기준점 0.5를 **결정 경계(Decision Boundary)**라고 합니다.

</br>

In [7]:
# TODO 3: 입력(X), 가중치(W), 편향(b)를 입력받는 predict_probability 함수를 정의해봅시다.

# XW + B = y
def predict_probability(X, W, b):
    z = X @ W + b 
    return sigmoid(z)



In [8]:
# TODO 4:
#  - X_sample에 X_train의 첫 번째 샘플을 2차원 행렬로 저장해봅시다.
x_sample = X_train[0:1]
print(x_sample)

#  - W에 피처 수만큼 0으로 초기화해봅시다.
W = np.zeros(X_train.shape[1])

#  - b에 0.0을 저장해봅시다.
b=0.0


[[ 0.79566902 -0.13197948  0.8195957   1.05393502]]


In [9]:
# TODO 5: predict_probability 함수를 이용해 결과를 예측해봅시다.
predict = predict_probability(x_sample, W, b)
print(predict)

[0.5]


</br>

# Binary Cross-Entropy 손실 함수 (BCE)
> 예측 확률과 실제 라벨 간의 차이를 측정하는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">분류 전용 손실 함수</mark>입니다.

$$L = -\frac{1}{m}\sum_{i=1}^{m}\left[y_i \log(\hat{p}_i) + (1 - y_i)\log(1 - \hat{p}_i)\right]$$

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">기호</th>
      <th>의미</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">$m$</td><td>샘플 수</td></tr>
    <tr><td style="text-align:center">$y_i$</td><td>실제 라벨 (0 또는 1)</td></tr>
    <tr><td style="text-align:center">$\hat{p}_i$</td><td>예측 확률 ($\sigma(z_i)$)</td></tr>
  </tbody>
</table>

</br>

## BCE의 직관적 이해

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">실제 라벨 $y$</th>
      <th style="text-align:center">예측 확률 $\hat{p}$</th>
      <th style="text-align:center">손실</th>
      <th>해석</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">1</td><td style="text-align:center">0.99</td><td style="text-align:center">0.01</td><td>정확한 예측 (낮은 손실)</td></tr>
    <tr><td style="text-align:center">1</td><td style="text-align:center">0.01</td><td style="text-align:center">4.61</td><td>잘못된 예측 (높은 손실)</td></tr>
    <tr><td style="text-align:center">0</td><td style="text-align:center">0.01</td><td style="text-align:center">0.01</td><td>정확한 예측 (낮은 손실)</td></tr>
    <tr><td style="text-align:center">0</td><td style="text-align:center">0.99</td><td style="text-align:center">4.61</td><td>잘못된 예측 (높은 손실)</td></tr>
  </tbody>
</table>

💡왜 MSE 대신 BCE를 사용할까?
> 시그모이드 + MSE 조합은 그래디언트가 매우 작아져 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">학습이 느려지는 문제</mark>가 있습니다.</br>
> BCE는 확률이 틀릴수록 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">벌점이 기하급수적으로 커져</mark> 빠른 학습이 가능합니다.

</br>

In [13]:
# TODO 6: binary_cross_entropy 함수를 정의해봅시다.

def binary_cross_entropy(y, p, epsilon=1e-12):

    # TODO 6-1: log(0) 방지를 위해 p 값을 epsilon부터 1-epsilon으로 제한해봅시다.
    p = np.clip(p, epsilon, 1-epsilon)
    # TODO 6-2: loss에 손실을 계산하여 저장해봅시다.
    loss = -np.mean(y * np.log(p) + (1.0-y) *np.log(1.0-p))
    # TODO 6-3. loss를 반환해봅시다.
    return loss

In [14]:
# TODO 7: 현재 가중치(W, b)로 훈련 데이터의 예측 확률을 구하고, BCE 손실을 계산해봅시다.
p_train = predict_probability(X_train, W, b)
loss = binary_cross_entropy(y_train, p_train)

print(p_train[:5])
print(y_train[:5])

[0.5 0.5 0.5 0.5 0.5]
[1 0 1 0 0]


💡수치 안정성
> `np.log(0)`은 $-\infty$를 반환합니다.</br>
> `eps=1e-12`를 더해 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">log(0) 에러를 방지</mark>합니다.</br>
> `np.clip(p, eps, 1-eps)`로 확률값을 안전한 범위로 제한합니다.

</br>

## 그래디언트 계산 (Gradient)
> 손실 함수를 파라미터로 미분하여 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">업데이트 방향과 크기</mark>를 결정합니다.

## $W$에 대한 미분

연쇄 법칙(Chain Rule)을 적용합니다: $\frac{\partial L}{\partial W} = \frac{\partial L}{\partial \hat{p}} \cdot \frac{\partial \hat{p}}{\partial z} \cdot \frac{\partial z}{\partial W}$

**Step 1.** BCE를 $\hat{p}$로 미분:

$$\frac{\partial L}{\partial \hat{p}_i} = -\frac{1}{m}\left(\frac{y_i}{\hat{p}_i} - \frac{1 - y_i}{1 - \hat{p}_i}\right)$$

**Step 2.** 시그모이드를 $z$로 미분:

$$\frac{\partial \hat{p}}{\partial z} = \sigma(z)(1 - \sigma(z)) = \hat{p}(1 - \hat{p})$$

**Step 3.** $z = XW + b$를 $W$로 미분:

$$\frac{\partial z}{\partial W} = X$$

**Step 1 × Step 2 (정리):**

$$\frac{\partial L}{\partial \hat{p}_i} \cdot \frac{\partial \hat{p}}{\partial z} = -\frac{1}{m}\left(\frac{y_i}{\hat{p}_i} - \frac{1 - y_i}{1 - \hat{p}_i}\right) \cdot \hat{p}_i(1 - \hat{p}_i) = \frac{1}{m}(\hat{p}_i - y_i)$$

**최종 결과 (× Step 3):**

$$\boxed{\nabla_W = \frac{1}{m} X^T (\hat{p} - y)}$$

## $b$에 대한 미분

$W$와 동일한 과정이지만, $z = XW + b$를 $b$로 미분하면:

$$\frac{\partial z}{\partial b} = 1$$

따라서 Step 3만 달라집니다:

$$\boxed{\nabla_b = \frac{1}{m} \sum_{i=1}^{m} (\hat{p}_i - y_i)}$$

In [15]:
# TODO 8: 가중치와 편향의 그래디언트를 계산하는 compute_gradients 함수를 정의해봅시다.

def compute_gradients(X, y, p):

    # TODO 8-1: 샘플 수를 m에 저장해봅시다.
    m = len(y)

    # TODO 8-2: 예측과 실제의 차이(p - y)를 diff에 저장해봅시다.
    diff = p-y

    # TODO 8-3: gradient_W에 가중치 기울기를 계산해봅시다.
    gradient_W = (1/m) * X.T @ diff

    # TODO 8-4: gradient_b에 편향 기울기를 계산해봅시다.
    gradient_b = (1/m) * np.sum(diff)
    # TODO 8-5: gradient_W와 gradient_b를 반환해봅시다.
    return gradient_W, gradient_b

💡선형 회귀와 동일한 구조
> 그래디언트 공식이 선형 회귀의 $\frac{1}{m}X^T(X\theta - y)$와 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">같은 형태</mark>입니다.</br>
> 차이점은 $\hat{p}$가 시그모이드를 통과한 확률이라는 점뿐입니다.

</br>

In [16]:
# TODO 9:
#  - m, n에 X_train의 샘플 수와 피처 수를 저장해봅시다.
m, n = X_train.shape

#  - W를 피처 수만큼 0으로 초기화해봅시다.
W = np.zeros(n)

#  - b에 0.0을 저장해봅시다.
b = 0.0

#  - learning_rate에 0.1을 저장해봅시다.
learning_rate = 0.1

#  - epochs에 300을 저장해봅시다.
epochs = 300;

In [18]:
# TODO 10: 순전파 → 손실 계산 → 역전파 → 파라미터 갱신 학습 루프를 구현해봅시다. *** 중요 *** 
# 순전파는 값을 넣어서 확인하는거 = y값이 나옴 = 실제의 정답 y값과 비교해서 손실값을 계산  = 역전파 = 가중치 갱신해서 줄여나가기
# 

for epoch in range(epochs):

    # TODO 10-1: predict_probability 함수로 예측 확률을 계산해봅시다.
    p = predict_probability(X_train, W, b)

    # TODO 10-2: binary_cross_entropy 함수로 BCE 손실을 계산해봅시다.
    loss = binary_cross_entropy(y_train, p)

    # TODO 10-3: compute_gradients 함수로 그래디언트를 계산해봅시다.
    gradient_W, gradient_b = compute_gradients(X_train, y_train, p)

    # TODO 10-4: 학습률을 적용하여 W와 b를 갱신해봅시다.
    W -= learning_rate * gradient_W
    b -= learning_rate * gradient_b

    if epoch % 50 == 0:
        print(epoch, loss)

0 0.6931471805599453
50 0.33046790347598465
100 0.2800196653976741
150 0.2539909958738605
200 0.23641628643438253
250 0.22296105602427405


</br>

# 정확도 (Accuracy)
> 모델 성능을 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">직관적으로 평가</mark>하는 지표입니다.

$$\text{Accuracy} = \frac{\text{올바르게 예측한 수}}{\text{전체 샘플 수}}$$

In [20]:
# TODO 11: 정확도를 계산하는 accuracy 함수를 정의하고, Train과 Valid 정확도를 출력해봅시다.

def accuracy(X, W, b, y):

    # TODO 11-1: predict_probability 함수로 예측 확률을 구하고, 0.5 이상이면 True로 분류해봅시다.
    prediction = predict_probability(X, W, b) >= 0.5

    # TODO 11-2: 예측과 정답을 비교하여 평균 정확도를 반환해봅시다.
    return np.mean(prediction == y)


print(accuracy(X_train, W, b, y_train))

print(accuracy(X_valid, W, b, y_valid))

0.925
1.0
